# φ^∞ Financial Chaos Stabilization Demonstration

This notebook demonstrates **Resonance-based volatility stabilization** using the 
C9 Chaotic Attractor Detection framework.

**Key Properties:**
- Real-time C9 chaotic attractor detection in price time-series
- QRT-envelope confidence scoring for mean-reversion targets
- Exact-φ Fibonacci retracement levels (not standard approximations)
- Phase recommendations: ACCUMULATE / HOLD / STABILIZE

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from phi_infinity_lattice_compression.financial_chaos import (
    ResonanceVolatilityStabilizer,
    VolatilityResonanceDetector,
)

plt.style.use("dark_background")
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=["#00f0ff", "#ffd700", "#ff3366", "#77dd77"])

## 1. Simulating a Volatile Market

We generate a realistic price series with:
- Geometric Brownian motion base
- Volatility clustering (GARCH-like regime switches)
- 500 ticks at ~1-minute HFT resolution

In [ ]:
np.random.seed(71)  # Stability-stable seed (C9 root 8)
n_ticks = 500
price = 100.0
prices = [price]

# Simulate regime-switching volatility
vol_regime = 0.01  # Low vol
for t in range(1, n_ticks):
    # Regime switch every ~80 ticks
    if t % 80 == 0:
        vol_regime = np.random.choice([0.005, 0.01, 0.04, 0.08])
    price *= 1 + np.random.normal(0.0001, vol_regime)
    prices.append(price)

fig, ax = plt.subplots()
ax.plot(prices, color="#00f0ff", linewidth=0.8)
ax.set_xlabel("Tick", fontsize=12)
ax.set_ylabel("Price", fontsize=12)
ax.set_title("Simulated HFT Price Series (Regime-Switching Volatility)", fontsize=14)
for t in range(80, n_ticks, 80):
    ax.axvline(x=t, color="#ffd700", alpha=0.3, linestyle="--")
plt.tight_layout()
plt.show()

## 2. Chaotic Attractor Detection

Feed the price series through the `VolatilityResonanceDetector` and visualize which 
windows enter the C9 chaotic zone.

In [ ]:
detector = VolatilityResonanceDetector(window_size=32)

results = []
for p in prices:
    r = detector.ingest(p)
    results.append(r)

# Extract data
c9_indices = [r["c9_index"] for r in results]
is_chaotic = [r["is_chaotic"] for r in results]
volatilities = [r["volatility"] for r in results]
recommendations = [r["recommendation"] for r in results]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Price with chaotic zones highlighted
axes[0].plot(prices, color="#00f0ff", linewidth=0.8)
for i, chaotic in enumerate(is_chaotic):
    if chaotic:
        axes[0].axvspan(i - 0.5, i + 0.5, color="#ff3366", alpha=0.15)
axes[0].set_ylabel("Price", fontsize=12)
axes[0].set_title("Price with Chaotic Zones (red shading)", fontsize=14)

# C9 indices over time
colors_c9 = ["#ff3366" if d in (3, 6, 9) else "#77dd77" for d in c9_indices]
axes[1].scatter(range(len(c9_indices)), c9_indices, c=colors_c9, s=4, alpha=0.7)
axes[1].axhline(y=3, color="#ff3366", alpha=0.3, linestyle="--")
axes[1].axhline(y=6, color="#ff3366", alpha=0.3, linestyle="--")
axes[1].axhline(y=9, color="#ff3366", alpha=0.3, linestyle="--")
axes[1].set_ylabel("C9 Index", fontsize=12)
axes[1].set_title("Resonant C9 Index of Volatility Scalar", fontsize=14)
axes[1].set_yticks(range(1, 10))

# Volatility
axes[2].fill_between(range(len(volatilities)), volatilities, color="#ffd700", alpha=0.5)
axes[2].plot(volatilities, color="#ffd700", linewidth=0.8)
axes[2].set_xlabel("Tick", fontsize=12)
axes[2].set_ylabel("Rolling Volatility", fontsize=12)
axes[2].set_title("Rolling Volatility (σ of log returns)", fontsize=14)

plt.tight_layout()
plt.show()

chaos_ratio = detector.get_chaos_ratio()
print(f"Overall chaos ratio: {chaos_ratio:.2%}")

## 3. Phase Recommendation Analysis

In [ ]:
from collections import Counter

rec_counts = Counter(recommendations)
labels = list(rec_counts.keys())
sizes = list(rec_counts.values())
color_map = {"ACCUMULATE": "#77dd77", "HOLD": "#ffd700", "STABILIZE": "#ff3366"}
colors = [color_map.get(label_name, "#888") for label_name in labels]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(
    sizes,
    labels=labels,
    colors=colors,
    autopct="%1.1f%%",
    startangle=140,
    textprops={"fontsize": 12, "color": "white"},
)
axes[0].set_title("Phase Recommendation Distribution", fontsize=14)

# Phase over time
phase_num = {"ACCUMULATE": 1, "HOLD": 2, "STABILIZE": 4}
phase_vals = [phase_num.get(p, 0) for p in recommendations]
phase_colors = [color_map.get(p, "#888") for p in recommendations]
axes[1].scatter(range(len(phase_vals)), phase_vals, c=phase_colors, s=5, alpha=0.7)
axes[1].set_yticks([1, 2, 4])
axes[1].set_yticklabels(["ACCUMULATE", "HOLD", "STABILIZE"])
axes[1].set_xlabel("Tick", fontsize=12)
axes[1].set_title("Phase Recommendation Over Time", fontsize=14)

plt.tight_layout()
plt.show()

## 4. Mean-Reversion Target & Fibonacci Levels

Using the `ResonanceVolatilityStabilizer` to compute reversion targets and 
exact-φ Fibonacci retracement levels.

In [ ]:
# Compute mean-reversion for the last window
last_window = prices[-32:]
last_vol = volatilities[-1]

target, confidence = ResonanceVolatilityStabilizer.compute_reversion_target(last_window, last_vol)
print(f"Current price:       {prices[-1]:.4f}")
print(f"Reversion target:    {target:.4f}")
print(f"Confidence (Damping):    {confidence:.4f}")

# Fibonacci levels from recent swing
recent_high = max(prices[-100:])
recent_low = min(prices[-100:])
levels = ResonanceVolatilityStabilizer.fibonacci_support_levels(prices[-1], recent_high, recent_low)

print("\nExact-φ Fibonacci Retracement Levels:")
print(f"{'Level':<14} {'Price':<12}")
print("-" * 28)
for name, val in levels.items():
    print(f"{name:<14} {val:<12.4f}")

In [ ]:
# Visualize Fibonacci levels on price chart
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(range(len(prices) - 100, len(prices)), prices[-100:], color="#00f0ff", linewidth=1.2)

fib_colors = ["#ff3366", "#ffd700", "#ff9922", "#77dd77", "#00f0ff", "#9977ff", "#ff3366"]
for (name, val), col in zip(levels.items(), fib_colors, strict=False):
    ax.axhline(y=val, color=col, alpha=0.6, linestyle="--", linewidth=1)
    ax.text(len(prices) - 100 + 2, val, f"{name} ({val:.2f})", fontsize=10, color=col)

ax.axhline(
    y=target,
    color="#77dd77",
    alpha=0.8,
    linestyle="-",
    linewidth=2,
    label=f"Reversion Target ({target:.2f})",
)
ax.set_xlabel("Tick", fontsize=12)
ax.set_ylabel("Price", fontsize=12)
ax.set_title("Exact-φ Fibonacci Retracement with Mean-Reversion Target", fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Summary

| Property | Result |
|----------|--------|
| Attractor detection | ✓ C9 digital root flagging |
| Phase recommendations | ✓ ACCUMULATE / HOLD / STABILIZE |
| Mean-reversion | ✓ Damping-envelope confidence scoring |
| Fibonacci levels | ✓ Exact-φ (not 0.618 approximation) |
| Streaming compatible | ✓ Single-tick ingestion |

*φ^∞ Lattice Framework — Financial Chaos Stabilization Protocol*